<a href="https://colab.research.google.com/github/eteitelbaum/code-satp/blob/main/training-and-inference-notebooks/training_location_extraction_t5_base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install --upgrade transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 65.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.46.2
    Uninstalling transformers-4.46.2:
      Successfully uninstalled transformers-4.46.2


In [ ]:
!pip install datasets evaluate rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 13.3 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=39cade996644bf9e16de85a3c80f120a2d7a8feacbfb5d7345ba3d066dd6881a
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dep

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

# Load your dataset
data = pd.read_csv('/content/drive/MyDrive/SATP_data/satp_location_matched.csv', dtype=str)

def prepare_data(df):
    inputs = [f"Extract the location of the incident: {summary}" for summary in df['incident_summary']]
    targets = df['Matched_Location'].tolist()
    return {'input_text': inputs, 'target_text': targets}

dataset = Dataset.from_dict(prepare_data(data))

# Split into train, validation, and test sets (80% train, 10% val, 10% test)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
test_val_split = dataset['test'].train_test_split(test_size=0.5, seed=42)

# Merge splits into a DatasetDict
dataset = DatasetDict({
    'train': dataset['train'],
    'validation': test_val_split['train'],
    'test': test_val_split['test']
})

print(dataset)



DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 7936
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 992
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 993
    })
})


In [ ]:
from transformers import T5Tokenizer

# Initialize the tokenizer
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)

# # Tokenization function
# def preprocess_function(examples):
#     # Tokenize the input and target text
#     model_inputs = tokenizer(
#         examples['input_text'],
#         max_length=512,
#         truncation=True,
#         padding="max_length"
#     )
#     labels = tokenizer(
#         examples['target_text'],
#         max_length=64,
#         truncation=True,
#         padding="max_length"
#     ).input_ids

#     # # Replace padding token id's in the labels with -100
#     # labels = [
#     #     [(l if l != tokenizer.pad_token_id else -100) for l in label]
#     #     for label in labels
#     # ]
#     model_inputs["labels"] = labels
#     return model_inputs

def preprocess_function(examples):
    inputs = examples['input_text']
    targets = examples['target_text']

    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding='max_length')

    # Tokenize targets with the `text_target` keyword argument
    labels = tokenizer(targets, max_length=64, truncation=True, padding='max_length')

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Apply tokenization
tokenized_datasets = dataset.map(preprocess_function, batched=True)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/7936 [00:00<?, ? examples/s]

Map:   0%|          | 0/992 [00:00<?, ? examples/s]

Map:   0%|          | 0/993 [00:00<?, ? examples/s]

In [ ]:
import torch
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Calculate exact match
    exact_matches = [
        int(pred.strip() == label.strip()) for pred, label in zip(decoded_preds, decoded_labels)
    ]
    accuracy = sum(exact_matches) / len(exact_matches)

    return {"exact_match": accuracy * 100}  # Return as percentage


In [ ]:
from transformers import T5ForConditionalGeneration, TrainingArguments, Trainer
import evaluate

# Initialize the model
model = T5ForConditionalGeneration.from_pretrained(model_name)
model.to(device)

# Define compute metrics
# rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in labels as we can't decode them
    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE scores
    # rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels)

    # Compute BLEU scores
    bleu_result = bleu.compute(predictions=decoded_preds, references=[[ref] for ref in decoded_labels])

    # Combine results
    metrics = {
        # "rouge1": rouge_result["rouge1"].mid.fmeasure * 100,
        # "rouge2": rouge_result["rouge2"].mid.fmeasure * 100,
        # "rougeL": rouge_result["rougeL"].mid.fmeasure * 100,
        "bleu": bleu_result["bleu"] * 100,
    }
    return metrics

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",          # Output directory for checkpoints and logs
    evaluation_strategy="epoch",    # Evaluate at the end of every epoch
    learning_rate=5e-5,             # Learning rate
    per_device_train_batch_size=8,  # Batch size for training
    per_device_eval_batch_size=4,   # Batch size for evaluation
    num_train_epochs=3,             # Number of training epochs
    weight_decay=0.01,              # Weight decay for regularization
    save_strategy="epoch",          # Save the model at each epoch
    logging_dir="./logs",           # Directory for storing logs
    logging_steps=100,              # Log every 100 steps
    load_best_model_at_end=True, # Load the best model at the end of training
    report_to='none'
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    # compute_metrics=compute_metrics
)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:

# Train the model
trainer.train()


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.102400,0.072874
2,0.075700,0.064537
3,0.069600,0.062928


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=2976, training_loss=0.1319156018316105, metrics={'train_runtime': 3585.3145, 'train_samples_per_second': 6.64, 'train_steps_per_second': 0.83, 'total_flos': 1.449806945845248e+16, 'train_loss': 0.1319156018316105, 'epoch': 3.0})

In [ ]:
# Save the model and tokenizer
model.save_pretrained('/content/drive/MyDrive/SATP_data/location_context_extraction/t5base_finetuned_model')
tokenizer.save_pretrained('/content/drive/MyDrive/SATP_data/location_context_extraction/t5base_finetuned_model')


('/content/drive/MyDrive/SATP_data/location_context_extraction/t5base_finetuned_model/tokenizer_config.json',
 '/content/drive/MyDrive/SATP_data/location_context_extraction/t5base_finetuned_model/special_tokens_map.json',
 '/content/drive/MyDrive/SATP_data/location_context_extraction/t5base_finetuned_model/spiece.model',
 '/content/drive/MyDrive/SATP_data/location_context_extraction/t5base_finetuned_model/added_tokens.json')

In [ ]:
# Evaluate the model on the test set
eval_results = trainer.evaluate(eval_dataset=tokenized_datasets["test"])
print(f"Test Set Evaluation Results: {eval_results}")


Test Set Evaluation Results: {'eval_loss': 0.066614530980587, 'eval_runtime': 47.7579, 'eval_samples_per_second': 20.792, 'eval_steps_per_second': 5.214, 'epoch': 3.0}


In [ ]:
def predict_location(summary):
    # Prepare the input text
    input_text = f"Extract the location of the incident: {summary}"
    input_ids = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).input_ids

    # Generate predictions
    outputs = model.generate(input_ids, max_length=512, num_beams=4, early_stopping=True)

    # Decode the output
    predicted_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return predicted_text

model.to('cpu')
# Example usage
incident_summary = "Three Communist Party of India-Maoist (CPI-Maoist) cadres, identified as Madvi Bhima (52), Madkam Hidma alias Sai Denga (33), and Paadam Ayate (24), surrendered in Sukma District in the Bastar division of Chhattisgarh on September 19, reports devdiscourse.com. Bhima was the ‘president’ of the Nilamadgu Revolutionary People’s Committee (RPC) Dandakaranya Adivasi Kisan Mazdoor Sangthan (DAKMS) unit of the Maoist group and carried a INR 100,000 reward. Hidma served as ‘vice-president’, while Ayate was the ‘head’ of Palachalma RPC 'Jantana Sarkar' (people's government of the Maoists), each also carrying a INR 100,000 reward. The trio cited their disillusionment with Maoist atrocities and were impressed by the state's Naxalite [Left Wing Extremist, LWE] elimination policy and welfare schemes as reasons for their surrender."
predicted_locations = predict_location(incident_summary)
print(f"Predicted Locations: {predicted_locations}")


Predicted Locations: chhattisgarh, sukma


In [ ]:
import torch
# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the model and tokenizer
model_path = '/content/drive/MyDrive/SATP_data/location_context_extraction/t5base_finetuned_model'
model = T5ForConditionalGeneration.from_pretrained(model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)
model.to(device)
print(f'Using device: {device}')


Using device: cuda
Using device: cuda


In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

# Load your dataset
data = pd.read_csv('/content/drive/MyDrive/SATP_data/satp_location_matched.csv', dtype=str)

def prepare_data(df):
    inputs = [f"Extract the location of the incident: {summary}" for summary in df['incident_summary']]
    targets = df['Matched_Location'].tolist()
    return {'input_text': inputs, 'target_text': targets}

dataset = Dataset.from_dict(prepare_data(data))

# Split into train, validation, and test sets (80% train, 10% val, 10% test)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
test_val_split = dataset['test'].train_test_split(test_size=0.5, seed=42)

# Merge splits into a DatasetDict
dataset = DatasetDict({
    'train': dataset['train'],
    'validation': test_val_split['train'],
    'test': test_val_split['test']
})

def preprocess_function(examples):
    inputs = examples['input_text']
    targets = examples['target_text']

    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding='max_length')

    # Tokenize targets with the `text_target` keyword argument
    labels = tokenizer(targets, max_length=64, truncation=True, padding='max_length')

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Apply tokenization
tokenized_dataset_test = dataset["test"].map(preprocess_function, batched=True)
# Set format for PyTorch
tokenized_dataset_test.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)


Map:   0%|          | 0/993 [00:00<?, ? examples/s]

In [ ]:
import numpy as np

def compute_metrics(predictions, labels):
    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = [
        tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
        for label in labels
    ]

    # Calculate exact match
    exact_matches = [
        int(pred.strip() == label.strip()) for pred, label in zip(decoded_preds, decoded_labels)
    ]
    accuracy = sum(exact_matches) / len(exact_matches)

    return {"exact_match": accuracy * 100}  # Return percentage


In [ ]:
import torch
from torch.utils.data import DataLoader

def evaluate_model(model, dataloader, tokenizer):
    model.eval()  # Set the model to evaluation mode
    all_predictions = []
    all_labels = []

    with torch.no_grad():  # Disable gradient computation
        for batch in dataloader:
            # Move input data to the appropriate device
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]

            # Generate predictions
            outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_length=64
            )
            predictions = outputs.cpu().numpy()  # Move predictions to CPU
            all_predictions.extend(predictions)
            all_labels.extend(labels.numpy())  # Move labels to CPU

    # Compute metrics
    metrics = compute_metrics(all_predictions, all_labels)
    return metrics


In [ ]:
from torch.utils.data import DataLoader

# Convert the test dataset to a PyTorch DataLoader
test_loader = DataLoader(tokenized_dataset_test, batch_size=8)


In [ ]:
# Evaluate the model on the test set
metrics = evaluate_model(model, test_loader, tokenizer)
print(f"Test Set Metrics: {metrics}")


Test Set Metrics: {'exact_match': 38.26787512588117}


In [ ]:
!pip install evaluate nltk rouge_score

In [ ]:
import evaluate

# Load BLEU and ROUGE evaluators
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

def compute_metrics(predictions, labels):
    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = [
        tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
        for label in labels
    ]

    # Format labels for BLEU (it expects a list of lists for references)
    formatted_labels = [[label] for label in decoded_labels]

    # Compute BLEU
    bleu = bleu_metric.compute(predictions=decoded_preds, references=formatted_labels)

    # Compute ROUGE
    rouge = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)

    # Combine and return metrics
    metrics = {
        "bleu": bleu["bleu"] * 100,  # Convert to percentage
        "rouge1": rouge["rouge1"] * 100,
        "rouge2": rouge["rouge2"] * 100,
        "rougeL": rouge["rougeL"] * 100,
    }
    return metrics


In [ ]:
def evaluate_model(model, dataloader, tokenizer):
    model.eval()  # Set the model to evaluation mode
    all_predictions = []
    all_labels = []

    with torch.no_grad():  # Disable gradient computation
        for batch in dataloader:
            # Move input data to the appropriate device
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]

            # Generate predictions
            outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_length=64
            )
            predictions = outputs.cpu().numpy()  # Move predictions to CPU
            all_predictions.extend(predictions)
            all_labels.extend(labels.cpu().numpy())  # Move labels to CPU

    # Compute metrics
    metrics = compute_metrics(all_predictions, all_labels)
    return metrics


In [ ]:
# Evaluate the model on the test set
metrics = evaluate_model(model, test_loader, tokenizer)
print(f"Test Set Metrics: {metrics}")


Test Set Metrics: {'bleu': 56.719800859549785, 'rouge1': 79.64346307374664, 'rouge2': 50.416606668577025, 'rougeL': 78.9309779679869}


In [ ]:
def compute_metrics(predictions, labels):
    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = [
        tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
        for label in labels
    ]

    # Exact Match
    exact_matches = [
        int(pred.strip() == label.strip()) for pred, label in zip(decoded_preds, decoded_labels)
    ]
    exact_match_score = sum(exact_matches) / len(exact_matches)

    # Recall
    recalls = []
    for pred, label in zip(decoded_preds, decoded_labels):
        pred_components = set(pred.strip().split(", "))
        label_components = set(label.strip().split(", "))
        recall = len(pred_components & label_components) / len(label_components)
        recalls.append(recall)
    avg_recall = sum(recalls) / len(recalls)

    return {
        "exact_match": exact_match_score * 100,
        "recall": avg_recall * 100,
    }


In [ ]:
metrics = evaluate_model(model, test_loader, tokenizer)
print(f"Test Set Metrics: {metrics}")


Test Set Metrics: {'exact_match': 38.26787512588117, 'recall': 87.71231957032548}


In [ ]:
import pandas as pd
import torch

# Initialize the model for evaluation
model.eval()

# Lists to store the results
incident_summaries = []
predicted_locations = []
actual_locations = []

# Evaluation loop
with torch.no_grad():
    for batch in test_loader:
        # Move input data to the appropriate device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]

        # Generate predictions
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_length=64
        )
        predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        labels_decoded = [
            tokenizer.decode([l for l in label if l != -100], skip_special_tokens=True)
            for label in labels
        ]

        # Append data to lists
        incident_summaries.extend(
            tokenizer.batch_decode(input_ids, skip_special_tokens=True)
        )
        predicted_locations.extend(predictions)
        actual_locations.extend(labels_decoded)

# Create a DataFrame
test_results_df = pd.DataFrame({
    "Incident Summary": incident_summaries,
    "Predicted Location": predicted_locations,
    "Actual Location": actual_locations,
})

# Save the DataFrame to a CSV file (optional)
test_results_df.to_csv('/content/drive/MyDrive/SATP_data/location_extraction_test_results.csv', index=False)

# Display the DataFrame
test_results_df.head()


,Incident Summary,Predicted Location,Actual Location
0,Extract the location of the incident: CPI-Maoi...,aurangabad,"aurangabad, state irrigation department"
1,Extract the location of the incident: CPI-Maoi...,"koraput, pottangi, pottangi","koraput, pottangi, pottangi, pottangi"
2,Extract the location of the incident: Police a...,Unknown,Unknown
3,Extract the location of the incident: Three pe...,"rohtas, barcha","rohtas, barcha"
4,Extract the location of the incident: Police r...,"latehar, manika, kumrahi forest","latehar, manika, kumrahi, manika"


In [ ]:
import requests
import pandas as pd
from gliner import GLiNER

# Load a pre-trained NER model
model = GLiNER.from_pretrained("urchade/gliner_base")

# Google Maps API key
API_KEY = userdata.get('googlemapsAPI')
GEOCODE_URL = "https://maps.googleapis.com/maps/api/geocode/json"

# Function to extract locations using NER
def extract_locations(text):
    """Extracts location entities from the given text using GLiNER."""
    labels = ["location"]  # Focus on location entities
    entities = model.predict_entities(text, labels)
    locations = [entity["text"] for entity in entities if entity["label"] == "location"]
    return locations

# Function to get location details using Google Geocoding API
def get_location_details(locations):
    """Given a list of location names, constructs a query, calls the Google Geocoding API,
    and returns state, district, subdistrict, and town/village (if any)."""
    query = ', '.join(locations)
    params = {
        'address': query,
        'key': API_KEY
    }
    response = requests.get(GEOCODE_URL, params=params)
    if response.status_code != 200:
        print(f"Error in API call: {response.status_code}")
        return None

    data = response.json()
    if data['status'] != 'OK':
        print(f"Geocoding API error: {data['status']}")
        return None

    # Parse the address components
    results = data.get('results', [])
    if not results:
        print("No results found.")
        return None

    # Take the first result
    address_components = results[0]['address_components']

    # Initialize components
    state = district = subdistrict = town_village = None

    # Map Google address types to desired components
    for component in address_components:
        types = component['types']
        if 'administrative_area_level_1' in types:
            state = component['long_name']
        elif 'administrative_area_level_2' in types:
            district = component['long_name']
        elif 'administrative_area_level_3' in types:
            subdistrict = component['long_name']
        elif 'locality' in types:
            town_village = component['long_name']
        elif 'sublocality' in types and not town_village:
            town_village = component['long_name']

    return {
        'state': state,
        'district': district,
        'subdistrict': subdistrict,
        'town_village': town_village
    }

# Load the incident data
satp_dat = pd.read_csv('/content/drive/MyDrive/SATP_data/satp_location_id.csv', dtype=str)
satp_data = satp_dat.head(100).copy()

satp_data['extracted_state'] = None
satp_data['extracted_district'] = None
satp_data['extracted_subdistrict'] = None
satp_data['extracted_town_village'] = None


# Process each summary in the data
for index, row in satp_data.iterrows():
    summary = row['incident_summary']  # Adjust column name if necessary
    locations = extract_locations(summary)
    satp_data.at[index, 'extracted_locations'] = locations
    if locations:
        location_details = get_location_details(locations)
        if location_details:
            # Update the DataFrame with location details
            satp_data.at[index, 'extracted_state'] = location_details.get('state')
            satp_data.at[index, 'extracted_district'] = location_details.get('district')
            satp_data.at[index, 'extracted_subdistrict'] = location_details.get('subdistrict')
            satp_data.at[index, 'extracted_town_village'] = location_details.get('town_village')
    else:
        print(f"No locations found in summary at index {index}")

# Display the updated DataFrame
print(satp_data.head())


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Geocoding API error: ZERO_RESULTS
Geocoding API error: ZERO_RESULTS
  incident_number           state        district                    block  \
0       101010701  Andhra Pradesh       Hyderabad  Gachibowli (Rangareddy)   
1       101010901  Andhra Pradesh       Nizamabad                      NaN   
2       101030601  Andhra Pradesh         Khammam                      NaN   
3       101051602  Andhra Pradesh  Vishakhapatnam                      NaN   
4       101060701  Andhra Pradesh   Visakhapatnam                GK Veedhi   

   village_name     other_areas     constituency  \
0           NaN       Cyberabad  Serilingampally   
1     Kamareddy             NaN        Kamareddy   
2  Bhadrachalam             NaN     Bhadrachalam   
3           NaN  Visakha Agency         Rayadurg   
4  Teegalabanda      Pedalavasa         Rayadurg   

                                    incident_summary extracted_state  \
0  An alleged arms supplier to the Communist Part...       Telangana   
1  A K